In [1]:
import os, signal, sys, time

sys.path.append(r'C:\Users\adria\coding\katja\DRL-in-international-economy-ai-economist-')

from ai_economist import foundation

In [2]:
from rllib.env_wrapper import RLlibEnvWrapper


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
c:\Users\adria\anaconda3\envs\ai-economist\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
c:\Users\adria\anaconda3\envs\ai-economist\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in

In [3]:
env_config_dict = {
    # Scenario with 2 planners
    "scenario_name": "custom/splitworld_overlay_regional",

        'components': [
        # (1) Building houses
        ('Build', {
            'skill_dist':                   'pareto', 
            'payment_max_skill_multiplier': 3,
            'build_labor':                  10,
            'payment':                      10
        }),
        # (2) Trading collectible resources
        ('ContinuousDoubleAuction', {
            'max_bid_ask':    10,
            'order_labor':    0.25,
            'max_num_orders': 5,
            'order_duration': 50
        }),
        # (3) Movement and resource collection
        ('Gather', {
            'move_labor':    0.2, #prev 1
            'collect_labor': 1,
            'skill_dist':    'pareto'
        }),
        # (4) Planner

        ("RegionalPeriodicBracketTax", {
            "region": "top",
            "planner_id": "p_top",
            "period": 100,
            "bracket_spacing": "us-federal",
            "usd_scaling": 1000,
            "disable_taxes": False,
        }),
        ("RegionalPeriodicBracketTax", {
            "region": "bottom",
            "planner_id": "p_bottom",
            "period": 100,
            "bracket_spacing": "us-federal",
            "usd_scaling": 1000,
            "disable_taxes": False,
        })

    ],
    

    # Map settings
    "env_layout_file": "map_100x50_water_gaps_3percent_resources.txt",
    "world_size": [100, 50],
    'episode_length': 1000, # Number of timesteps per episode

    'starting_agent_coin': 10,
    'fixed_four_skill_and_loc': True,

    # Agents and planners
    "n_agents": 4,
    "planner_subclasses": ["TopPlanner", "BottomPlanner"],
    "planner_ids": ["p_top", "p_bottom"],
    "planner_classes": ["TopPlanner", "BottomPlanner"],

    # Modes
    "multi_action_mode_planner": True,
    'multi_action_mode_agents': False,

    'flatten_observations': True,
    # When Flattening masks, concatenate each action subspace mask into a single array.
    # Note: flatten_masks = True is required for masking action logits in the code below.
    'flatten_masks': True,
    
    # How often to save the dense logs
    'dense_log_frequency': 1
}

In [4]:
env_obj = RLlibEnvWrapper({"env_config_dict": env_config_dict}, verbose=True)

ValueError: need at least one array to concatenate

In [ ]:
obs_space_p_top     = env_obj.observation_space_pl["p_top"]
act_space_p_top     = env_obj.action_space_pl["p_top"]

obs_space_p_bottom  = env_obj.observation_space_pl["p_bottom"]
act_space_p_bottom  = env_obj.action_space_pl["p_bottom"]

obs_space_agent     = env_obj.observation_space["0"]
act_space_agent     = env_obj.action_space["0"]


In [ ]:
import ray
from ray.rllib.agents.ppo import PPOTrainer

In [ ]:
# Policies
policies = {
    # Mobile agents (shared policy)
    "a": (
        None,
        env_obj.observation_space,   # dict-like space for any mobile agent (e.g., "0")
        env_obj.action_space,
        {}
    ),
    # Top planner
    "p_top": (
        None,
        env_obj.observation_space_pl["p_top"],
        env_obj.action_space_pl["p_top"],
        {}
    ),
    # Bottom planner
    "p_bottom": (
        None,
        env_obj.observation_space_pl["p_bottom"],
        env_obj.action_space_pl["p_bottom"],
        {}
    ),
}

# Mapping: digits -> "a"; exact planner ids -> their own policies
policy_mapping_fun = lambda i: (
    "a" if str(i).isdigit()
    else ("p_top" if i == "p_top" else ("p_bottom" if i == "p_bottom" else "a"))
)

policies_to_train = ["a", "p_top", "p_bottom"]



In [ ]:
# policies = {
#     "a": (
#         None,
#         env_obj.observation_space,
#         env_obj.action_space,
#         {}
#     ),
#     "p_top": (
#         None,
#         env_obj.observation_space_pl,   # single planner space
#         env_obj.action_space_pl,
#         {}
#     ),
#     "p_bottom": (
#         None,
#         env_obj.observation_space_pl,   # same as above
#         env_obj.action_space_pl,
#         {}
#     ),
# }

# policy_mapping_fun = lambda i: (
#     "a" if str(i).isdigit()
#     else ("p_top" if i == "p_top" else ("p_bottom" if i == "p_bottom" else "a"))
# )

# policies_to_train = ["a", "p_top", "p_bottom"]

In [ ]:
trainer_config = {
    "multiagent": {
        "policies": policies,
        "policies_to_train": policies_to_train,
        "policy_mapping_fn": policy_mapping_fun,
    }
}

trainer_config.update(
    {
        "num_workers": 2,
        "num_envs_per_worker": 2,
        # Other training parameters
        "train_batch_size":  4000,
        "sgd_minibatch_size": 4000,
        "num_sgd_iter": 1
    }
)

In [ ]:
# We also add the "num_envs_per_worker" parameter for the env. wrapper to index the environments.
env_config = {
    "env_config_dict": env_config_dict,
    "num_envs_per_worker": trainer_config.get('num_envs_per_worker'),   
}

trainer_config.update(
    {
        "env_config": env_config        
    }
)

In [ ]:
config = {
    "env": RLlibEnvWrapper,
    "env_config": {"env_config_dict": env_config_dict},
    "num_envs_per_worker": trainer_config.get("num_envs_per_worker", 1),
}

In [ ]:
ray.init(webui_host="127.0.0.1")

In [ ]:
trainer = PPOTrainer(
    env=RLlibEnvWrapper,
    config=trainer_config,
    )